# StartLux 27B 轨迹 → Qwen3.6-27B 上的 LoRA（GPU：RunPod、Vast.ai、AutoDL、Colab 等）

用 StartLux 27B 的 Agent 轨迹，在它的训练底座 `Qwen/Qwen3.6-27B` 上训练一个 LoRA。轨迹来自公开的 MCP-Universe 评测日志（ModelScope `j772190144/MCP-universe_log_for_6_models`，Apache-2.0）：日志里的模型名是 `eval_27b_mcpunivse_260622`，六个领域的历史评分与 StartLux 官网公布的 StartLux-V1.0-27B-Preview 成绩逐项一致（日志没有权重哈希，不能证明权重完全相同）。`convert_mcp_universe.py` 把原始日志转成本 notebook 读的格式。导出的 adapter 交给融合 notebook（`Qwen38_pi_AuxMix_e15_FP8.ipynb`）作为辅料 `StartLuxLoRA`，占用原来留给 StartLux 的位置（权重 4，系数 0.4）。

**流程**：锁定底座 commit → 读轨迹、按评分筛选、用底座自己的 chat template 渲染完整对话 → 只在助手回合（含思考内容和工具调用）上算损失 → 4-bit（或 BF16）底座 + LoRA 训练，断点存在工作目录（Colab 上是 Drive）→ 训练前后各算一次留出集损失 → 导出 adapter 和清单。

**为什么不合并成完整模型**：LoRA 的增量很小（单个权重常在 1e-5～1e-4），合并进 BF16 权重再减回底座，得到的大多是舍入噪声。融合 notebook 直接读 adapter，逐矩阵用 float32 算 scale·(B @ A) 加进 D，不上传 55GB。

**和融合 notebook 的衔接**：融合 notebook 在 Colab CPU 上读 Drive 里的 adapter 目录（三个文件，约 1GB）。在 Colab 上训练时，本 notebook 最后直接打印要填的 `TRACE_LORA_DIR`。在别的平台上训练时，填 `EXPORT_REPO`，训完自动上传到你的私有 Hugging Face 仓库，并打印一段在 Colab 里把它下载到 Drive 的代码；也可以手动下载这三个文件再传到 Drive。之后在融合 notebook 第 1 节填 `TRACE_LORA_DIR`，在 CPU 高 RAM 运行时上从头运行：它会和 DavidAU 一起补加到已经算好的 D 上，然后搜索、上传。

**运行时**：一张 80GB 显卡（H100、A100 80GB、A800 80GB、H800）或更大（H20 96GB）。默认 4-bit 底座（QLoRA，NF4），80GB 卡上每条轨迹能到 32k token；40GB 卡（Colab 的 A100 一般是这种）只能到 12k，能用的轨迹少很多。磁盘要放底座约 55GB，连同数据、断点和导出，工作目录留 100GB。任何能开 Jupyter 的 Linux GPU 机器都行：不在 Colab 上时不挂载 Drive，所有文件放在 `WORK_DIR`（RunPod / Vast.ai 默认 `/workspace`，AutoDL 默认数据盘 `/root/autodl-tmp`）。国内平台连不上 huggingface.co 时，把 `HF_ENDPOINT` 设成镜像（如 `https://hf-mirror.com`），或先打开平台自带的学术加速。

**用时**：取决于轨迹的 token 数。第 4 节准备好数据后会打印总 token 数和粗略估计；训练时每一步按实测速度打印剩余时间。断开后从头运行全部单元格，会从 Drive 上最近的断点接着训。

**注意**：真实的 27B 权重、Qwen3.6 的线性注意力层、GPU 上的训练都没有在这里运行过；纯计算部分和整条流程只在 CPU 上用极小的模型测过（`tests/test_trace_lora.py`）。第一次请先设 `SMOKE_TEST = True` 跑 2 步确认能通，再正式训练。


## 1. 配置

只改下面这一格。

| 项 | 说明 |
|---|---|
| `WORK_DIR` | 数据、底座、断点和导出都在这里。Colab 上是 Drive 的记录目录；RunPod / Vast.ai 上是 `/workspace/qwen38_trace`；AutoDL 上是 `/root/autodl-tmp/qwen38_trace` |
| `DATA_FILES` | 轨迹文件，`.jsonl`（一行一条）或 `.json`（列表），可写通配符。默认是 `convert_mcp_universe.py` 转出的 `startlux_mcp_universe_runs.jsonl`（StartLux 27B 在 MCP-Universe 上的 746 次运行，其中 289 次通过），解压后放到 `WORK_DIR/startlux_traces/` |
| `HF_ENDPOINT` | 留空用 huggingface.co；国内平台可填 `https://hf-mirror.com` |
| `EXPORT_REPO` | 不在 Colab 上训练时建议填一个新的私有仓库名（如 `你的用户名/startlux-trace-lora`）：训完把 adapter 传上去，方便融合 notebook 在 Colab 里取用。token 要有写权限 |
| `DATA["score_field"]` / `["min_score"]` | 按评分筛选。`auto` 依次找 `reward`、`score`、`strict_pass`、`passed`… 也可写点路径（如 `"grades.score"`）。默认只用评分 ≥ 1 的（严格通过）；`None` 全部使用，包括答错的，不建议 |
| `DATA["max_tokens"]` | 渲染后超过这个长度的整条丢掉（不截断：截掉的工具返回可能正是回答引用的内容）。`auto`：显存 ≥ 75 GiB 时 4-bit 用 32768、BF16 用 16384；40GB 卡用 12288。通过的 289 次运行里，≤12k 的 108 次，≤16k 的 161 次，≤32k 的 241 次 |
| `DATA["tools"]` | `records`：用每条记录自带的工具定义，没有就按它实际的调用补一个最简定义（只有参数名和类型，不编造说明）；`synthesize`：所有记录共用补出的定义 |
| `LORA` | 秩、alpha、加在哪些模块（解码层的 `self_attn`、`linear_attn`、`mlp`；视觉、MTP、嵌入、输出头不加） |
| `TRAIN` | 学习率、轮数、每步几条轨迹、每几步存一次断点；`max_steps_per_session` 限制本次会话最多训几步 |
| `PRECISION` | 默认 `4bit`（QLoRA：底座约 16GB，80GB 卡上能训到 32k，多用一半的轨迹，效果通常与 BF16 的 LoRA 接近）；`bf16` 只到 16k；`auto` 按显存选 |
| `SMOKE_TEST` | `True`：只训 2 步、不导出，用单独的目录，确认能跑通 |

每条记录要有完整的多轮消息（OpenAI 格式：`role`、`content`、助手的 `tool_calls` 与 `reasoning_content`、`tool` 角色的返回）。消息放在 `messages`、`trajectory`、`conversation` 之一里。


In [ ]:
import os, sys, json
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 工作目录：Colab 上是 Drive（与融合 notebook 相同的记录目录）；RunPod / Vast.ai 用持久卷 /workspace；AutoDL 用数据盘
if IN_COLAB:
    WORK_DIR = "/content/drive/MyDrive/qwen38_efficientthink"
else:
    WORK_DIR = next((d for d in ("/workspace", "/root/autodl-tmp") if os.path.isdir(d)), str(Path.home())) + "/qwen38_trace"
TRACE_DIR = WORK_DIR + "/trace_lora"                                            # 数据、断点和导出
DATA_FILES = [WORK_DIR + "/startlux_traces/startlux_mcp_universe_runs.jsonl"]   # convert_mcp_universe.py 的输出（解压后放这里）
HF_ENDPOINT = ""     # 国内平台连不上 huggingface.co 时填 "https://hf-mirror.com"
EXPORT_REPO = ""     # 不在 Colab 上训练时建议填：训完把 adapter 上传到这个私有仓库（如 "你的用户名/startlux-trace-lora"）
if HF_ENDPOINT:
    os.environ["HF_ENDPOINT"] = HF_ENDPOINT          # 要在导入 huggingface_hub 之前设

BASE_REPO = "Qwen/Qwen3.6-27B"   # 产生轨迹的检查点的训练底座
BASE_REVISION = ""               # 留空：第一次运行时锁定当时的 commit（记在 TRACE_DIR/base_pin.json）
LOCAL_BASE = "/content/trace_lora_base" if IN_COLAB else WORK_DIR + "/base_qwen36"   # 底座（约 55GB）

DATA = {
    "score_field": "auto",
    "min_score": 1.0,            # None：不筛选
    "max_tokens": "auto",        # auto：4-bit 且显存 ≥ 75 GiB 时 32768，其余 16384；超过的整条丢掉
    "tools": "records",
    "mask_empty_think": True,
    "template_kwargs": {},
    "eval_fraction": 0.05, "eval_max": 64, "seed": 0,
}
LORA = {"rank": 32, "alpha": 32, "dropout": 0.0, "parts": ["self_attn", "linear_attn", "mlp"]}
TRAIN = {"lr": 1e-4, "epochs": 2, "accumulation": 8, "weight_decay": 0.0, "warmup_fraction": 0.05, "floor_ratio": 0.1,
         "clip": 1.0, "chunk": 2048, "save_every": 10, "seed": 0, "max_steps_per_session": None}
PRECISION = "4bit"           # 4bit（默认，80GB 卡上能训到 32k，多用一半的轨迹）、bf16（只到 16k）或 auto
ATTN = "sdpa"
INSTALL = True
SMOKE_TEST = False


## 2. 环境

挂载 Drive，检查 GPU，安装依赖：transformers（≥ 5.8，Qwen3.5 系列的模型类从这一版开始有）、peft、accelerate；4-bit 时加 bitsandbytes。线性注意力的快速内核 `flash-linear-attention` 和 `causal-conv1d` 也会尝试安装；装不上时 transformers 退回纯 torch 实现，能跑但更慢、更占显存，会打印提示。


In [ ]:
import shutil, subprocess, sys

if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive as _drive
    _drive.mount("/content/drive")
Path(TRACE_DIR).mkdir(parents=True, exist_ok=True)
_free = shutil.disk_usage(TRACE_DIR).free / 1024**3
print(f"工作目录 {WORK_DIR}，剩余 {_free:.0f} GiB" + ("（建议至少 100 GiB：底座约 55GB）" if _free < 100 else ""))

def _gpu_gib():
    exe = shutil.which("nvidia-smi")
    if not exe:
        return 0.0
    out = subprocess.run([exe, "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
    try:
        return float(out.stdout.strip().splitlines()[0]) / 1024
    except (ValueError, IndexError):
        return 0.0

GPU_GIB = _gpu_gib()
if GPU_GIB == 0:
    raise RuntimeError("没有 GPU：请换一台有 80GB 显卡（H100、A100 80GB、A800、H800）的机器后从头运行。")
_need_4bit = PRECISION == "4bit" or (PRECISION == "auto" and GPU_GIB < 75)

def _pip(*args, fatal=True, timeout=None):
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"提示：pip install {' '.join(args)} 超过 {timeout} 秒还没装完（多半在现场编译），跳过。")
        return False
    if result.returncode != 0:
        message = f"pip install {' '.join(args)} 失败：{(result.stderr or result.stdout)[-600:]}"
        if fatal:
            raise RuntimeError(message)
        print("提示：" + message)
    return result.returncode == 0

if INSTALL:
    _pip("-U", "transformers>=5.8", "peft", "accelerate", "safetensors", "huggingface_hub")
    if _need_4bit:
        _pip("-U", "bitsandbytes")
    FAST_LINEAR_ATTENTION = _pip("flash-linear-attention", fatal=False, timeout=900)
    FAST_CONV = _pip("causal-conv1d", "--no-build-isolation", fatal=False, timeout=600)
    if not (FAST_LINEAR_ATTENTION and FAST_CONV):
        print("提示：线性注意力的快速内核没有装全，transformers 会用纯 torch 实现：能跑，但更慢、更占显存。")

import importlib.metadata as _md
import torch
VERSIONS = {p: _md.version(p) for p in ("torch", "transformers", "peft", "accelerate", "safetensors", "huggingface_hub")}
for _opt in ("bitsandbytes", "flash-linear-attention", "causal-conv1d"):
    try:
        VERSIONS[_opt] = _md.version(_opt)
    except _md.PackageNotFoundError:
        pass
print("依赖：", VERSIONS)
_props = torch.cuda.get_device_properties(0)
print(f"GPU：{_props.name}，显存 {_props.total_memory / 1024**3:.1f} GiB")

TOK = os.environ.get("HF_TOKEN", "")
if not TOK:
    try:
        from google.colab import userdata
        TOK = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if not TOK and not IN_COLAB:
    from getpass import getpass
    TOK = getpass("Hugging Face token（要能读 Qwen/Qwen3.6-27B；填 EXPORT_REPO 时还要有写权限）: ")
TOK = TOK.strip()
from huggingface_hub import HfApi
api = HfApi(token=TOK or None)


## 3. 模块

纯计算部分（读记录、规整消息、工具定义、渲染、损失落在哪些 token 上、划分、导出检查）和 GPU 部分（装载、LoRA、分块损失、可续跑的训练、评估、导出）都在下面这个内嵌模块里。单元测试直接从本 notebook 取出它的源码。


In [ ]:
import importlib.util
EMBEDDED_DIR = Path(TRACE_DIR) / 'modules'
EMBEDDED_DIR.mkdir(parents=True, exist_ok=True)
def load_embedded_module(name, source):
    path = EMBEDDED_DIR / (name + '.py')
    path.write_text(source, encoding='utf-8')
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    globals()[name] = module
    return module

TRACE_SOURCE = r'''"""StartLux checkpoint traces -> a LoRA adapter on Qwen3.6-27B, exported for the fusion notebook's "lora" source.

Pure parts (reading and normalizing records, tool schemas, the chat rendering, which tokens carry the loss, the split,
the export check) need only the standard library and a tokenizer. The GPU parts (loading the base, the LoRA layers, the
chunked loss, the resumable training loop, evaluation, export) need torch, transformers and peft and are imported lazily.
"""
import bisect
import glob
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import time

VERSION = 'trace-lora-2026-11-02.1'
IM_START, IM_END = '<|im_start|>', '<|im_end|>'
EMPTY_THINK = re.compile(r'<think>\s*</think>\s*')
MESSAGE_KEYS = ('messages', 'trajectory', 'conversation', 'conversations')
SCORE_KEYS = ('reward', 'score', 'strict_pass', 'passed', 'pass', 'success', 'final_score')
# the fusion notebook's reader takes exactly these adapter tensor names (qmerge_pipeline._LORA_KEY)
ADAPTER_KEY = re.compile(r'^(?:base_model\.model\.)?(?P<module>.+)\.lora_(?P<part>[AB])(?:\.[A-Za-z0-9_]+)?\.weight$')
LAYER = re.compile(r'(?:^|\.)layers\.\d+\.')
SKIP_MODULE = re.compile(r'(?:^|\.)(?:visual|vision_tower|vision_model|mtp|nextn|lm_head|embed_tokens)(?:\.|$)')


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()


def sha_file(path, chunk=16 * 1024 ** 2):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


# ---------------------------------------------------------------------------------------------- reading records
def expand_paths(patterns):
    """Files named directly or by glob patterns, in a stable order, without duplicates."""
    out = []
    for pattern in patterns:
        hits = sorted(glob.glob(str(Path(pattern).expanduser()), recursive=True)) if any(c in str(pattern) for c in '*?[') else [str(Path(pattern).expanduser())]
        for hit in hits:
            if Path(hit).is_file() and hit not in out:
                out.append(hit)
    if not out:
        raise FileNotFoundError('没有找到数据文件：' + ', '.join(map(str, patterns)))
    return out


def read_records(paths):
    """[(record id, record)] from .jsonl files (one record per line) and .json files (a list, or a dict holding one
    under data / records / trajectories, or a single record)."""
    out = []
    for p in map(Path, paths):
        if p.suffix == '.jsonl':
            for k, line in enumerate(p.read_text(encoding='utf-8').splitlines(), 1):
                if line.strip():
                    out.append((f'{p.name}:{k}', json.loads(line)))
            continue
        data = json.loads(p.read_text(encoding='utf-8'))
        if isinstance(data, dict):
            data = next((data[k] for k in ('data', 'records', 'trajectories') if isinstance(data.get(k), list)), [data])
        for k, rec in enumerate(data):
            out.append((f'{p.name}#{k}', rec))
    return out


def find_messages(rec):
    for key in MESSAGE_KEYS:
        value = rec.get(key)
        if isinstance(value, dict):
            value = value.get('messages')
        if isinstance(value, list) and value and isinstance(value[0], dict) and 'role' in value[0]:
            return value
    raise ValueError('找不到消息列表（试过 ' + ', '.join(MESSAGE_KEYS) + '）')


def text_of(content):
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, dict) and 'text' in content:
        return str(content['text'])
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and part.get('type') in (None, 'text', 'input_text', 'output_text'):
                parts.append(str(part.get('text', '')))
            else:
                raise ValueError('消息里有非文本内容：' + str(part.get('type') if isinstance(part, dict) else type(part).__name__))
        return ''.join(parts)
    raise ValueError('不认识的 content：' + type(content).__name__)


def parse_arguments(raw):
    if isinstance(raw, dict):
        return raw
    if raw in (None, ''):
        return {}
    try:
        value = json.loads(raw)
    except (TypeError, ValueError):
        raise ValueError('工具调用的参数不是合法 JSON：' + str(raw)[:120]) from None
    if not isinstance(value, dict):
        raise ValueError('工具调用的参数不是 JSON 对象：' + str(raw)[:120])
    return value


def normalize_messages(messages):
    """OpenAI-style messages -> what chat templates take: role, content (text), reasoning_content, tool_calls
    [{type, id, function: {name, arguments: dict}}], tool messages with tool_call_id. Messages after the last assistant
    turn carry no loss and are dropped."""
    out = []
    for m in messages:
        role = 'tool' if m.get('role') == 'function' else m.get('role')
        if role not in ('system', 'user', 'assistant', 'tool'):
            raise ValueError('不认识的角色：' + str(m.get('role')))
        msg = {'role': role, 'content': text_of(m.get('content'))}
        if role == 'assistant':
            extra = m.get('provider_specific_fields') or {}
            reasoning = m.get('reasoning_content') or extra.get('reasoning_content') or m.get('reasoning') or ''
            if reasoning:
                msg['reasoning_content'] = str(reasoning)
            calls = []
            for c in m.get('tool_calls') or []:
                f = c.get('function') or c
                calls.append({'type': 'function', 'id': c.get('id'),
                              'function': {'name': str(f['name']), 'arguments': parse_arguments(f.get('arguments'))}})
            if calls:
                msg['tool_calls'] = calls
        elif role == 'tool':
            if m.get('tool_call_id'):
                msg['tool_call_id'] = m['tool_call_id']
            if m.get('name'):
                msg['name'] = m['name']
        out.append(msg)
    last = max((k for k, m in enumerate(out) if m['role'] == 'assistant'), default=None)
    if last is None:
        raise ValueError('没有助手回合')
    return out[:last + 1]


def normalize_tools(raw):
    """OpenAI ({type, function: {...}}), bare ({name, parameters}) or MCP ({name, inputSchema}) tool definitions ->
    OpenAI function schemas."""
    out = []
    for t in raw or []:
        f = t.get('function', t)
        params = f.get('parameters') or f.get('input_schema') or f.get('inputSchema') or {'type': 'object', 'properties': {}}
        out.append({'type': 'function', 'function': {'name': str(f['name']), 'description': str(f.get('description') or ''),
                                                     'parameters': params}})
    return out


def json_type(value):
    if isinstance(value, bool):
        return 'boolean'
    if isinstance(value, int):
        return 'integer'
    if isinstance(value, float):
        return 'number'
    if isinstance(value, list):
        return 'array'
    if isinstance(value, dict):
        return 'object'
    return 'string'


def synthesize_tools(conversations):
    """Minimal function schemas from the calls actually made, for records that carry no tool definitions: each tool's
    parameters with the JSON types seen, required = those present in every call. No descriptions are invented."""
    seen = {}
    for msgs in conversations:
        for m in msgs:
            for c in m.get('tool_calls', []):
                f = c['function']
                entry = seen.setdefault(f['name'], {'calls': 0, 'params': {}})
                entry['calls'] += 1
                for k, v in f['arguments'].items():
                    p = entry['params'].setdefault(k, {'types': set(), 'count': 0})
                    p['types'].add(json_type(v))
                    p['count'] += 1
    tools = []
    for name in sorted(seen):
        e = seen[name]
        props = {}
        for k in sorted(e['params']):
            types = sorted(e['params'][k]['types'])
            props[k] = {'type': types[0] if len(types) == 1 else types}
        required = [k for k in sorted(e['params']) if e['params'][k]['count'] == e['calls']]
        tools.append({'type': 'function', 'function': {'name': name, 'description': '',
                                                       'parameters': {'type': 'object', 'properties': props, 'required': required}}})
    return tools


def as_score(value):
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    if isinstance(value, (int, float)):
        return float(value) if math.isfinite(value) else None
    if isinstance(value, str):
        low = value.strip().lower()
        if low in ('pass', 'passed', 'true', 'yes', 'success'):
            return 1.0
        if low in ('fail', 'failed', 'false', 'no', 'failure'):
            return 0.0
        try:
            return float(low)
        except ValueError:
            return None
    return None


def record_score(rec, field='auto'):
    """The record's score: a dotted path (e.g. "grades.score"), or with "auto" the first of SCORE_KEYS present."""
    if field != 'auto':
        value = rec
        for part in field.split('.'):
            value = value.get(part) if isinstance(value, dict) else None
        return as_score(value)
    for key in SCORE_KEYS:
        if key in rec:
            score = as_score(rec[key])
            if score is not None:
                return score
    return None


def extract_uploaded(wanted, places, target):
    """Find `wanted` (a file name) directly or inside a .zip uploaded to one of `places`, and put it into `target`.
    Returns the path, or None. Lets a Jupyter upload of the zip be used as is."""
    import zipfile
    target = Path(target)
    if (target / wanted).is_file():
        return target / wanted
    for place in map(Path, places):
        if not place.is_dir():
            continue
        if (place / wanted).is_file():
            target.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(place / wanted, target / wanted)
            return target / wanted
        for archive in sorted(place.glob('*.zip')):
            try:
                with zipfile.ZipFile(archive) as z:
                    member = next((n for n in z.namelist() if n.split('/')[-1] == wanted), None)
                    if member:
                        target.mkdir(parents=True, exist_ok=True)
                        with z.open(member) as src, open(target / wanted, 'wb') as dst:
                            shutil.copyfileobj(src, dst)
                        return target / wanted
            except zipfile.BadZipFile:
                continue
    return None


# ---------------------------------------------------------------------------------------------- rendering and labels
def render(tokenizer, messages, tools=None, template_kwargs=None):
    return tokenizer.apply_chat_template(messages, tools=tools or None, tokenize=False, add_generation_prompt=False,
                                         **(template_kwargs or {}))


def assistant_spans(text, mask_empty_think=True):
    """Character spans of the assistant turns in a ChatML rendering: from just after '<|im_start|>assistant\\n' to the
    end of that turn's '<|im_end|>' (so the model also learns to stop). An empty <think></think> scaffold the template
    inserts by itself is left out: the traces never show the model writing it."""
    spans, head, pos = [], IM_START + 'assistant\n', 0
    while True:
        a = text.find(head, pos)
        if a < 0:
            return spans
        start = a + len(head)
        end = text.find(IM_END, start)
        if end < 0:
            raise ValueError('助手回合没有结束标记 <|im_end|>')
        end += len(IM_END)
        if mask_empty_think:
            m = EMPTY_THINK.match(text, start)
            if m:
                start = m.end()
        spans.append((start, end))
        pos = end


def label_mask(offsets, spans):
    """Per token (char start, char end): whether it lies wholly inside one of the spans (sorted, disjoint)."""
    starts = [s for s, _ in spans]
    out = []
    for a, b in offsets:
        k = bisect.bisect_right(starts, a) - 1
        out.append(b > a and k >= 0 and b <= spans[k][1])
    return out


def encode(tokenizer, text, spans):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(enc['input_ids'])
    mask = label_mask([tuple(o) for o in enc['offset_mapping']], spans)
    return ids, [t if m else -100 for t, m in zip(ids, mask)]


def prepare(records, tokenizer, opts):
    """records [(id, record)] -> (examples [{id, input_ids, labels, score}], report). opts: score_field, min_score
    (None keeps every record), max_tokens, mask_empty_think, template_kwargs, tools ('records', 'synthesize')."""
    report = {'records': len(records), 'no_messages': 0, 'bad_records': [], 'below_min_score': 0, 'no_score': 0,
              'too_long': 0, 'no_labels': 0, 'kept': 0, 'scores': {}}
    convs = []
    for rid, rec in records:
        try:
            msgs = normalize_messages(find_messages(rec))
        except ValueError as error:
            report['no_messages'] += 1
            if len(report['bad_records']) < 10:
                report['bad_records'].append(f'{rid}: {error}')
            continue
        score = record_score(rec, opts.get('score_field', 'auto'))
        key = 'none' if score is None else f'{score:g}'
        report['scores'][key] = report['scores'].get(key, 0) + 1
        if opts.get('min_score') is not None:
            if score is None:
                report['no_score'] += 1
                continue
            if score < float(opts['min_score']):
                report['below_min_score'] += 1
                continue
        convs.append((rid, rec, msgs, score))
    shared = synthesize_tools([c[2] for c in convs]) if opts.get('tools', 'records') == 'synthesize' else None
    examples, lengths, labelled = [], [], 0
    for rid, rec, msgs, score in convs:
        tools = shared if shared is not None else (normalize_tools(rec.get('tools')) or synthesize_tools([msgs]))
        text = render(tokenizer, msgs, tools, opts.get('template_kwargs'))
        ids, labels = encode(tokenizer, text, assistant_spans(text, opts.get('mask_empty_think', True)))
        if len(ids) > int(opts['max_tokens']):
            report['too_long'] += 1
            continue
        n = sum(1 for t in labels if t != -100)
        if not n:
            report['no_labels'] += 1
            continue
        examples.append({'id': rid, 'input_ids': ids, 'labels': labels, 'score': score})
        lengths.append(len(ids))
        labelled += n
    report['kept'] = len(examples)
    report['tokens'] = sum(lengths)
    report['labelled_tokens'] = labelled
    if lengths:
        ls = sorted(lengths)
        report['length'] = {'median': ls[len(ls) // 2], 'p90': ls[int(0.9 * (len(ls) - 1))], 'max': ls[-1]}
    return examples, report


def split(examples, eval_fraction, eval_max, seed=0):
    """A fixed held-out set: the examples whose hashed id ranks first, at most eval_max of them (at least one when
    there are two or more examples)."""
    ranked = sorted(examples, key=lambda e: hashlib.sha256(f'{seed}:{e["id"]}'.encode()).hexdigest())
    n = min(int(eval_max), int(round(len(examples) * float(eval_fraction))))
    if len(examples) >= 2:
        n = max(1, n)
    held = {e['id'] for e in ranked[:n]}
    return [e for e in examples if e['id'] not in held], [e for e in examples if e['id'] in held]


def adapter_key_problems(keys):
    """Names the fusion notebook would refuse: anything not a lora_A / lora_B pair of a decoder-layer module."""
    problems, pairs = [], {}
    for key in keys:
        m = ADAPTER_KEY.match(key)
        if not m or not LAYER.search(m['module']) or SKIP_MODULE.search(m['module']):
            problems.append(key)
            continue
        pairs.setdefault(m['module'], set()).add(m['part'])
    problems += [f'{module}（只有 lora_{"".join(sorted(parts))}）' for module, parts in pairs.items() if parts != {'A', 'B'}]
    return problems


# ---------------------------------------------------------------------------------------------- the schedule
def learning_rate(step, total, peak, warmup_fraction=0.05, floor_ratio=0.1):
    """Linear warm-up, then cosine down to floor_ratio * peak, for optimizer step `step` (0-based) of `total`."""
    warm = max(1, int(round(total * warmup_fraction)))
    if step < warm:
        return peak * (step + 1) / warm
    progress = (step - warm) / max(1, total - warm)
    return peak * (floor_ratio + (1 - floor_ratio) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress))))


def step_plan(n_examples, accumulation, epochs, seed=0):
    """[[example indices of optimizer step k]]: each epoch a fresh fixed shuffle cut into groups of `accumulation`."""
    plan = []
    for epoch in range(int(epochs)):
        order = list(range(n_examples))
        random.Random(seed * 1000003 + epoch).shuffle(order)
        plan += [order[k:k + accumulation] for k in range(0, n_examples, accumulation)]
    return plan


# ---------------------------------------------------------------------------------------------- GPU side
def precision_for(total_gib, setting='auto'):
    """'bf16' (base weights in BF16, needs about 70 GiB at 16k tokens), '4bit' (NF4 QLoRA), or None (too small)."""
    if setting != 'auto':
        return setting
    if total_gib >= 75:
        return 'bf16'
    if total_gib >= 38:
        return '4bit'
    return None


def load_base(local_dir, precision='bf16', attn='sdpa'):
    """The base model on GPU 0. A natively multimodal checkpoint may only load through the image-text class; text-only
    training through it is fine. Returns (model, class name)."""
    import torch
    import transformers
    kwargs = dict(dtype=torch.bfloat16, device_map={'': 0}, attn_implementation=attn, low_cpu_mem_usage=True)
    if precision == '4bit':
        kwargs['quantization_config'] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=['lm_head', 'visual', 'mtp', 'embed_tokens'])
    errors = []
    for name in ('AutoModelForCausalLM', 'AutoModelForImageTextToText'):
        cls = getattr(transformers, name, None)
        if cls is None:
            continue
        try:
            return cls.from_pretrained(local_dir, **kwargs), name
        except (ValueError, KeyError) as error:
            errors.append(f'{name}: {str(error)[:300]}')
    raise RuntimeError('装载不了底座模型：' + ' | '.join(errors))


def lora_targets(model, parts=('self_attn', 'linear_attn', 'mlp')):
    """Full names of the linear layers inside the language model's decoder layers (attention, linear attention, MLP);
    vision, MTP, embeddings and the output head are left alone."""
    names = []
    for name, module in model.named_modules():
        if SKIP_MODULE.search(name) or not LAYER.search(name):
            continue
        if not any(f'.{p}.' in f'.{name}.' for p in parts):
            continue
        if type(module).__name__ in ('Linear', 'Linear4bit', 'Linear8bitLt') and hasattr(module, 'weight'):
            names.append(name)
    if not names:
        raise RuntimeError('没有找到可加 LoRA 的线性层（解码层里的 self_attn / linear_attn / mlp）')
    return names


def add_lora(model, targets, rank, alpha, dropout=0.0, quantized=False):
    import torch
    from peft import LoraConfig, get_peft_model
    if quantized:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={'use_reentrant': False})
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    if getattr(model, 'config', None) is not None:
        model.config.use_cache = False
    config = LoraConfig(r=int(rank), lora_alpha=float(alpha), lora_dropout=float(dropout), target_modules=list(targets),
                        bias='none', use_rslora=False, use_dora=False)
    model = get_peft_model(model, config)
    for p in model.parameters():                       # the adapter is trained and kept in float32
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    return model


def decoder_and_head(model):
    """(the language decoder returning last_hidden_state after the final norm, the output head) of a (PEFT-wrapped)
    causal or image-text model. The loss is computed from these, chunk by chunk, so the full logits never exist."""
    base = model.get_base_model() if hasattr(model, 'get_base_model') else model
    decoder = base.get_decoder() if hasattr(base, 'get_decoder') else None
    if decoder is None:
        decoder = getattr(getattr(base, 'model', None), 'language_model', None) or getattr(base, 'model', None)
    head = base.get_output_embeddings()
    if decoder is None or head is None:
        raise RuntimeError('找不到解码器或输出头')
    return decoder, head


def hidden_states(decoder, ids):
    out = decoder(input_ids=ids, use_cache=False)
    return out.last_hidden_state if hasattr(out, 'last_hidden_state') else out[0]


def chunked_loss(hidden, head, labels, chunk=2048):
    """(sum of next-token cross-entropies at the labelled positions, their count). The logits are made `chunk`
    positions at a time under activation checkpointing, so a [tokens, vocabulary] matrix never exists in full."""
    import torch
    import torch.nn.functional as F
    from torch.utils.checkpoint import checkpoint
    target = labels[:, 1:]
    h = hidden[:, :-1, :]
    where = (target != -100).nonzero(as_tuple=True)
    hs, ys = h[where], target[where]
    total = hs.new_zeros((), dtype=torch.float32)

    weight = getattr(head, 'weight', None)
    dtype = weight.dtype if weight is not None and weight.is_floating_point() else hs.dtype

    def piece(x, y):
        # the 4-bit preparation keeps the output head in float32 while the decoder hands over bfloat16
        return F.cross_entropy(head(x.to(dtype)).float(), y, reduction='sum')
    for s in range(0, hs.shape[0], chunk):
        total = total + checkpoint(piece, hs[s:s + chunk], ys[s:s + chunk], use_reentrant=False)
    return total, int(ys.numel())


def example_loss(decoder, head, example, device, chunk=2048):
    import torch
    ids = torch.tensor([example['input_ids']], device=device)
    labels = torch.tensor([example['labels']], device=device)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == 'cuda'):
        h = hidden_states(decoder, ids)
    return chunked_loss(h, head, labels, chunk)


def check_decoder(model, decoder, head, example, device, length=64):
    """The decoder + head route against the model's own logits on a short prefix: they must agree."""
    import torch
    ids = torch.tensor([example['input_ids'][:length]], device=device)
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == 'cuda'):
        mine = head(hidden_states(decoder, ids)).float()
        theirs = model(input_ids=ids, use_cache=False).logits.float()
    diff = (mine - theirs).abs().max().item()
    scale = theirs.abs().max().item() or 1.0
    return {'max_abs_diff': diff, 'relative': diff / scale, 'ok': diff / scale < 2e-2}


def evaluate(decoder, head, examples, device, chunk=2048):
    import torch
    total, count = 0.0, 0
    with torch.no_grad():
        for ex in examples:
            loss, n = example_loss(decoder, head, ex, device, chunk)
            total += float(loss)
            count += n
    return {'mean_loss': total / max(count, 1), 'tokens': count, 'examples': len(examples)}


def trainable_state(model):
    from peft import get_peft_model_state_dict
    return {k: v.detach().float().cpu().contiguous() for k, v in get_peft_model_state_dict(model).items()}


def save_checkpoint(directory, model, optimizer, step, identity):
    import torch
    from safetensors.torch import save_file
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    tmp = directory / f'ckpt-{step:06d}.tmp'
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir()
    save_file(trainable_state(model), str(tmp / 'adapter.safetensors'))
    torch.save(optimizer.state_dict(), tmp / 'optimizer.pt')
    atomic_json(tmp / 'state.json', {'step': step, 'identity': identity})
    final = directory / f'ckpt-{step:06d}'
    if final.exists():
        shutil.rmtree(final)
    tmp.replace(final)
    kept = sorted(p for p in directory.glob('ckpt-*') if p.is_dir() and re.fullmatch(r'ckpt-\d{6}', p.name))
    for old in kept[:-2]:                                  # keep the two newest
        shutil.rmtree(old)
    return final


def latest_checkpoint(directory, identity):
    for ckpt in sorted(Path(directory).glob('ckpt-[0-9]*'), reverse=True):
        try:
            state = json.loads((ckpt / 'state.json').read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if state.get('identity') == identity and (ckpt / 'adapter.safetensors').is_file() and (ckpt / 'optimizer.pt').is_file():
            return ckpt, state['step']
    return None, 0


def train(model, examples, opts, checkpoint_dir, identity, log=print):
    """The resumable loop. opts: lr, epochs, accumulation, weight_decay, warmup_fraction, floor_ratio, clip, chunk,
    save_every, seed, max_steps_per_session (stop after that many steps in this call, with a checkpoint). Each optimizer
    step averages the loss over all labelled tokens of its examples. Returns (history of this call, finished)."""
    import torch
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    device = next(p for p in model.parameters() if p.requires_grad).device
    decoder, head = decoder_and_head(model)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=float(opts['lr']), weight_decay=float(opts.get('weight_decay', 0.0)))
    plan = step_plan(len(examples), int(opts['accumulation']), int(opts['epochs']), int(opts.get('seed', 0)))
    ckpt, start = latest_checkpoint(checkpoint_dir, identity)
    if ckpt is not None:
        set_peft_model_state_dict(model, {k: v.to(device) for k, v in load_file(str(ckpt / 'adapter.safetensors')).items()})
        optimizer.load_state_dict(torch.load(ckpt / 'optimizer.pt', map_location=device))
        log(f'从断点 {ckpt.name} 继续：第 {start}/{len(plan)} 步之后')
    limit = opts.get('max_steps_per_session')
    end = len(plan) if not limit else min(len(plan), start + int(limit))
    model.train()
    history, started, seen_tokens = [], time.perf_counter(), 0
    for step in range(start, end):
        lr = learning_rate(step, len(plan), float(opts['lr']), float(opts.get('warmup_fraction', 0.05)), float(opts.get('floor_ratio', 0.1)))
        for group in optimizer.param_groups:
            group['lr'] = lr
        batch = [examples[i] for i in plan[step]]
        norm = sum(sum(1 for t in ex['labels'][1:] if t != -100) for ex in batch)
        total = 0.0
        for ex in batch:
            loss, n = example_loss(decoder, head, ex, device, int(opts.get('chunk', 2048)))
            (loss / max(norm, 1)).backward()
            total += float(loss.detach())
            seen_tokens += len(ex['input_ids'])
        grad_norm = float(torch.nn.utils.clip_grad_norm_(params, float(opts.get('clip', 1.0))))
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        elapsed = time.perf_counter() - started
        done = step + 1 - start
        record = {'step': step + 1, 'loss': total / max(norm, 1), 'lr': lr, 'grad_norm': grad_norm}
        history.append(record)
        log(f'第 {step + 1}/{len(plan)} 步：损失 {record["loss"]:.4f}，学习率 {lr:.2e}，梯度范数 {grad_norm:.3f}；'
            f'{seen_tokens / max(elapsed, 1e-9):.0f} token/s，估计还需 {elapsed / done * (len(plan) - step - 1) / 60:.0f} 分钟')
        if (step + 1) % int(opts.get('save_every', 10)) == 0 or step + 1 == end:
            save_checkpoint(checkpoint_dir, model, optimizer, step + 1, identity)
    model.eval()
    return history, end == len(plan)


def export_adapter(model, directory, manifest):
    """adapter_model.safetensors + adapter_config.json (PEFT layout) + trace_lora_manifest.json, checked against what
    the fusion notebook reads."""
    from safetensors import safe_open
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(directory), safe_serialization=True)
    path = directory / 'adapter_model.safetensors'
    with safe_open(str(path), framework='pt', device='cpu') as f:
        keys = list(f.keys())
    problems = adapter_key_problems(keys)
    if problems:
        raise RuntimeError('导出的 adapter 里有融合 notebook 不收的张量：' + ', '.join(problems[:5]))
    config = json.loads((directory / 'adapter_config.json').read_text(encoding='utf-8'))
    out = {**manifest, 'kind': 'trace_lora', 'version': VERSION, 'adapter_sha256': sha_file(path),
           'adapter_tensors': len(keys), 'rank': config.get('r'), 'lora_alpha': config.get('lora_alpha')}
    atomic_json(directory / 'trace_lora_manifest.json', out)
    return out
'''
load_embedded_module('trace_lora', TRACE_SOURCE)
print('trace_lora', trace_lora.VERSION)


## 4. 锁定底座、准备数据

先只下载底座的 tokenizer 和配置（几 MB），用它的 chat template 渲染每条轨迹。准备好的数据按“数据文件 + 设置 + chat template”的指纹存到 `TRACE_DIR/prepared/<指纹>/`，之后的会话直接沿用。


In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

PREC = trace_lora.precision_for(GPU_GIB, PRECISION)
if PREC is None:
    raise RuntimeError(f"显存只有 {GPU_GIB:.0f} GiB，不够训练 27B（至少要 40GB 的 A100）。")
if DATA["max_tokens"] == "auto":
    DATA["max_tokens"] = (32768 if PREC == "4bit" else 16384) if GPU_GIB >= 75 else 12288
print(f"底座精度 {PREC}；每条轨迹最长 {DATA['max_tokens']} token")

_pin_path = Path(TRACE_DIR) / "base_pin.json"
_pin = json.loads(_pin_path.read_text()) if _pin_path.exists() else None
if _pin and _pin["repo"] == BASE_REPO and (not BASE_REVISION or _pin["revision"] == BASE_REVISION):
    BASE_PIN = _pin
else:
    _info = api.model_info(BASE_REPO, revision=BASE_REVISION or None)
    BASE_PIN = {"repo": BASE_REPO, "revision": _info.sha}
    trace_lora.atomic_json(_pin_path, BASE_PIN)
print(f"底座：{BASE_PIN['repo']} @ {BASE_PIN['revision']}")

snapshot_download(BASE_PIN["repo"], revision=BASE_PIN["revision"], local_dir=LOCAL_BASE, token=TOK or None,
                  allow_patterns=["*.json", "*.jinja", "*.txt", "*.model", "tokenizer*"])
tokenizer = AutoTokenizer.from_pretrained(LOCAL_BASE)
if not getattr(tokenizer, "is_fast", False):
    raise RuntimeError("需要 fast tokenizer（要用 offset_mapping 判断哪些 token 属于助手回合）")

# 数据还没放到 DATA_FILES 时，在常见的上传位置找 startlux_mcp_universe_runs.jsonl 或装着它的 zip，解压过去
for _pattern in DATA_FILES:
    _want = Path(_pattern)
    if not any(c in _want.name for c in "*?[") and not _want.is_file():
        _found = trace_lora.extract_uploaded(_want.name, [WORK_DIR, str(Path(WORK_DIR).parent), os.getcwd(), str(Path.home()), "/content"], _want.parent)
        if _found:
            print("已从上传的文件里取出：", _found)
DATA_PATHS = trace_lora.expand_paths(DATA_FILES)
_data_id = {"files": [{"name": Path(p).name, "sha256": trace_lora.sha_file(p)} for p in DATA_PATHS],
            "data": DATA, "template_sha256": trace_lora.digest(tokenizer.chat_template), "base": BASE_PIN,
            "version": trace_lora.VERSION}
DATA_FP = trace_lora.digest(_data_id)[:16]
PREP_DIR = Path(TRACE_DIR) / "prepared" / DATA_FP
if (PREP_DIR / "report.json").exists():
    TRAIN_SET = [json.loads(l) for l in (PREP_DIR / "train.jsonl").read_text().splitlines()]
    EVAL_SET = [json.loads(l) for l in (PREP_DIR / "eval.jsonl").read_text().splitlines()]
    REPORT = json.loads((PREP_DIR / "report.json").read_text())
    print("沿用已准备好的数据：", PREP_DIR)
else:
    records = trace_lora.read_records(DATA_PATHS)
    examples, REPORT = trace_lora.prepare(records, tokenizer, DATA)
    TRAIN_SET, EVAL_SET = trace_lora.split(examples, DATA["eval_fraction"], DATA["eval_max"], DATA["seed"])
    REPORT.update(train=len(TRAIN_SET), eval=len(EVAL_SET), identity=_data_id)
    PREP_DIR.mkdir(parents=True, exist_ok=True)
    for _name, _rows in (("train.jsonl", TRAIN_SET), ("eval.jsonl", EVAL_SET)):
        (PREP_DIR / _name).write_text("".join(json.dumps(r) + "\n" for r in _rows))
    trace_lora.atomic_json(PREP_DIR / "report.json", REPORT)
    _first = examples[0] if examples else None
    if _first:   # one rendering to read: the loss falls on the [[ ]] parts
        _rec = dict(records)[_first["id"]]
        _msgs = trace_lora.normalize_messages(trace_lora.find_messages(_rec))
        _tools = (trace_lora.synthesize_tools([trace_lora.normalize_messages(trace_lora.find_messages(r)) for _, r in records])
                  if DATA["tools"] == "synthesize" else (trace_lora.normalize_tools(_rec.get("tools")) or trace_lora.synthesize_tools([_msgs])))
        _text = trace_lora.render(tokenizer, _msgs, _tools, DATA["template_kwargs"])
        print("示例渲染（前 3000 字符；[[ ]] 里是计损失的部分）：")
        _shown, _pos = [], 0
        for _a, _b in trace_lora.assistant_spans(_text, DATA["mask_empty_think"]):
            _shown += [_text[_pos:_a], "[[", _text[_a:_b], "]]"]
            _pos = _b
        print(("".join(_shown) + _text[_pos:])[:3000])

print(json.dumps({k: v for k, v in REPORT.items() if k != "identity"}, ensure_ascii=False, indent=2))
if not TRAIN_SET:
    raise RuntimeError("筛选后没有可训练的轨迹：看上面的 scores（各评分的条数）和 bad_records，调整 DATA 里的 score_field / min_score。")
_tokens = sum(len(e["input_ids"]) for e in TRAIN_SET) * TRAIN["epochs"]
print(f"训练 token 总数（含 {TRAIN['epochs']} 轮）：{_tokens / 1e6:.2f}M。粗略估计：H100 约 {_tokens / 2500 / 3600:.1f} 小时，"
      f"A100 80GB 约 {_tokens / 1500 / 3600:.1f} 小时，4-bit 再慢约一半；实际以训练时打印的速度为准。")


## 5. 装载底座并加 LoRA

下载完整底座（约 55GB）到本地盘，装到 GPU 上，在解码层的线性层上加 LoRA（adapter 用 float32）。装好后用一小段输入核对“解码器 + 输出头”这条路算出的 logits 与模型自己的一致：训练时的损失就按这条路分块算，从不展开完整的 [token 数, 词表] 矩阵。


In [ ]:
snapshot_download(BASE_PIN["repo"], revision=BASE_PIN["revision"], local_dir=LOCAL_BASE, token=TOK or None)
base_model, BASE_CLASS = trace_lora.load_base(LOCAL_BASE, PREC, ATTN)
TARGETS = trace_lora.lora_targets(base_model, tuple(LORA["parts"]))
print(f"底座按 {PREC} 装载（{BASE_CLASS}）；LoRA 加在 {len(TARGETS)} 个线性层上，例如 {TARGETS[:3]}")
torch.manual_seed(int(TRAIN["seed"]))
model = trace_lora.add_lora(base_model, TARGETS, LORA["rank"], LORA["alpha"], LORA["dropout"], quantized=(PREC == "4bit"))
model.print_trainable_parameters()
DEVICE = next(p for p in model.parameters() if p.requires_grad).device
DECODER, HEAD = trace_lora.decoder_and_head(model)
CHECK = trace_lora.check_decoder(model, DECODER, HEAD, (EVAL_SET or TRAIN_SET)[0], DEVICE)
print("解码器 + 输出头核对：", CHECK)
if not CHECK["ok"]:
    raise RuntimeError("按解码器 + 输出头算出的 logits 与模型自己的不一致，不能按这条路算损失。请把上面的输出反馈回来。")
print(f"显存已用 {torch.cuda.memory_allocated() / 1024**3:.1f} GiB")


## 6. 训练（可续跑）

每个优化步用 `accumulation` 条轨迹，按它们所有计损失的 token 平均。断点每 `save_every` 步存到 Drive（只留最近两份）；断开后从头运行全部单元格会从最近的断点继续（数据、底座、LoRA 和训练设置都相同时才续）。训练前先在留出集上算底座本身的损失，训完再算一次。


In [ ]:
_run_id = {"data": DATA_FP, "base": BASE_PIN, "lora": LORA, "precision": PREC, "smoke": SMOKE_TEST,
           "train": {k: v for k, v in TRAIN.items() if k != "max_steps_per_session"}, "version": trace_lora.VERSION}
RUN_ID = trace_lora.digest(_run_id)
RUN_DIR = Path(TRACE_DIR) / ("smoke" if SMOKE_TEST else "runs") / RUN_ID[:16]
RUN_DIR.mkdir(parents=True, exist_ok=True)
trace_lora.atomic_json(RUN_DIR / "run.json", _run_id)
_base_eval_path = RUN_DIR / "eval_base.json"
if _base_eval_path.exists():
    EVAL_BASE = json.loads(_base_eval_path.read_text())
elif EVAL_SET:
    with model.disable_adapter():
        EVAL_BASE = trace_lora.evaluate(DECODER, HEAD, EVAL_SET, DEVICE, TRAIN["chunk"])
    trace_lora.atomic_json(_base_eval_path, EVAL_BASE)
else:
    EVAL_BASE = None
print("留出集上底座本身的损失：", EVAL_BASE)

_opts = dict(TRAIN)
if SMOKE_TEST:
    _opts["max_steps_per_session"] = 2
HISTORY, FINISHED = trace_lora.train(model, TRAIN_SET, _opts, RUN_DIR / "checkpoints", RUN_ID)
with open(RUN_DIR / "history.jsonl", "a") as _f:
    for _h in HISTORY:
        _f.write(json.dumps(_h) + "\n")
if FINISHED and EVAL_SET:
    EVAL_AFTER = trace_lora.evaluate(DECODER, HEAD, EVAL_SET, DEVICE, TRAIN["chunk"])
    trace_lora.atomic_json(RUN_DIR / "eval_after.json", EVAL_AFTER)
    print("留出集损失：底座", EVAL_BASE["mean_loss"], "→ 加 LoRA 后", EVAL_AFTER["mean_loss"])
elif not FINISHED:
    print("本次会话的步数已用完；从头运行全部单元格会从断点继续。" + ("（冒烟测试：能跑通就把 SMOKE_TEST 改回 False 正式训练）" if SMOKE_TEST else ""))


## 7. 导出给融合 notebook

训练完成后导出到 `RUN_DIR/adapter/`：`adapter_model.safetensors`、`adapter_config.json`（PEFT 格式）和 `trace_lora_manifest.json`（底座 commit、数据文件与筛选、LoRA 与训练设置、留出集损失、adapter 的 SHA-256、依赖版本）。导出前核对每个张量名都是融合 notebook 收的“解码层模块的 lora_A / lora_B”。


In [ ]:
if SMOKE_TEST or not FINISHED:
    print("还没有导出：" + ("冒烟测试不导出。" if SMOKE_TEST else "训练还没完成。"))
else:
    import datetime
    _eval_after = json.loads((RUN_DIR / "eval_after.json").read_text()) if (RUN_DIR / "eval_after.json").exists() else None
    MANIFEST = trace_lora.export_adapter(model, RUN_DIR / "adapter", {
        "base_repo": BASE_PIN["repo"], "base_revision": BASE_PIN["revision"], "base_class": BASE_CLASS, "precision": PREC,
        "source_model": "eval_27b_mcpunivse_260622 (StartLux 27B on MCP-Universe; traces only, its weights were not available)",
        "data": {"files": _data_id["files"], "settings": DATA, "report": {k: v for k, v in REPORT.items() if k != "identity"}},
        "lora": LORA, "targets": len(TARGETS), "train": TRAIN, "steps": len(trace_lora.step_plan(len(TRAIN_SET), TRAIN["accumulation"], TRAIN["epochs"])),
        "eval": {"base": EVAL_BASE, "after": _eval_after}, "versions": VERSIONS,
        "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    })
    print(json.dumps({k: MANIFEST[k] for k in ("adapter_sha256", "adapter_tensors", "rank", "lora_alpha")}, indent=2))
    _adapter = RUN_DIR / "adapter"
    if EXPORT_REPO:
        api.create_repo(EXPORT_REPO, private=True, exist_ok=True)
        _commit = api.upload_folder(folder_path=str(_adapter), repo_id=EXPORT_REPO,
                                    commit_message=f"StartLux trace LoRA {MANIFEST['adapter_sha256'][:12]}")
        _rev = getattr(_commit, "oid", None) or api.model_info(EXPORT_REPO).sha
        print(f"\n已上传到私有仓库 {EXPORT_REPO} @ {_rev}")
        print("在融合 notebook（Colab CPU）里新开一个单元格运行下面两行，把它放到 Drive：")
        print(f'from huggingface_hub import snapshot_download\n'
              f'snapshot_download("{EXPORT_REPO}", revision="{_rev}", local_dir="/content/drive/MyDrive/qwen38_efficientthink/trace_lora/adapter", token=TOK)')
        print('然后在第 1 节填：TRACE_LORA_DIR = "/content/drive/MyDrive/qwen38_efficientthink/trace_lora/adapter"')
    elif IN_COLAB:
        print("\n把下面这一行填进融合 notebook 第 1 节，然后在 CPU 高 RAM 运行时上从头运行：")
        print(f'TRACE_LORA_DIR = "{_adapter}"')
    else:
        print(f"\nadapter 在 {_adapter}（三个文件）。把整个目录下载下来，传到 Drive 的 qwen38_efficientthink/trace_lora/adapter/，"
              "再在融合 notebook 第 1 节填 TRACE_LORA_DIR = \"/content/drive/MyDrive/qwen38_efficientthink/trace_lora/adapter\"。"
              "也可以填 EXPORT_REPO 后重新运行本格，自动上传到私有仓库。")


## 8. 说明

- **产物**：`TRACE_DIR/runs/<指纹>/adapter/`（约 1GB，float32）。`checkpoints/` 在导出之后可以删；`prepared/` 留着可以在改训练设置时省掉重新准备数据。
- **这项辅料的分量**：在融合里它的系数是 0.4，再乘上整个 D 的 λ（搜索范围 0.025–0.2），所以最终只有训出增量的约 1%–8% 进入成品，和原计划里 StartLux 的地位相同。
- **局限**：
  - 学的是这个检查点在这些任务上的行为，不是它的完整后训练；轨迹数量和覆盖面决定了它能带来多少。
  - 增量是在 Qwen3.6-27B 上学的，融合时原样移到 Qwen3.8 系的 pi 上（task arithmetic），跨一代会损失一部分效果。
  - 只用评分通过的轨迹时样本更少；放宽 `min_score` 会混进答错的回答。
- **需要你在 Colab 上看的**：
  1. 第 2 节：transformers 版本、线性注意力快速内核是否装上；
  2. 第 4 节：报告里的条数、各评分的分布、`too_long`、示例渲染里 `[[ ]]` 圈出的是不是助手的回答（含思考和工具调用）；
  3. 第 5 节：`BASE_CLASS`、LoRA 层数和例子、解码器核对是否 `ok`、显存；
  4. 第 6 节：前几步的 token/s 和估计剩余时间；损失是否下降；训完后留出集损失是否低于底座。
